# 13 — Informe reproducible previo al test

Genera tablas agregadas y figuras `ggplot2` para Sepsis-3 a seis horas usando exclusivamente development y validation. Compara prevalencia, regresión logística y gradient boosting bajo los mismos folds agrupados por paciente. El test permanece cerrado. En el demo, todos los resultados son controles técnicos, no estimaciones clínicas.

**Entradas:** artefactos validados de los pasos 09–12 y configuración versionada. **Salidas:** tablas agregadas de rendimiento, subgrupos y missingness, más figuras bajo `reports/figures/`. **Semilla:** `config/modeling.json`.

In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys
import pandas as pd
from IPython.display import SVG, display
PROJECT_ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(PROJECT_ROOT/'src')) if str(PROJECT_ROOT/'src') not in sys.path else None
sys.path.insert(0,str(PROJECT_ROOT)) if str(PROJECT_ROOT) not in sys.path else None
from mimic_sepsis.artifacts import ArtifactStore, ArtifactValidationError
from mimic_sepsis.modeling import assemble_modeling_table, make_gradient_boosting_pipeline
from mimic_sepsis.recalibration import calibration_readiness
from mimic_sepsis.report_artifacts import canonical_sha256, implementation_sha256, read_aggregate_report, write_aggregate_report
from mimic_sepsis.reporting import build_development_report
from scripts.build_demo_sofa_incremental import canonical_config
artifact_config=canonical_config()
model_config=json.loads((PROJECT_ROOT/'config/modeling.json').read_text())
evaluation_config=json.loads((PROJECT_ROOT/'config/evaluation.json').read_text())
subgroup_config=json.loads((PROJECT_ROOT/'config/subgroups.json').read_text())
calibration_config=json.loads((PROJECT_ROOT/'config/calibration.json').read_text())
boosting_config=model_config['gradient_boosting']
if boosting_config.get('early_stopping') is not False: raise ValueError('El comparador debe desactivar early stopping interno.')
nonlinear_pipeline=make_gradient_boosting_pipeline(model_config['clinical_baseline_features'],**{key:value for key,value in boosting_config.items() if key!='early_stopping'},seed=model_config['seed'])
valid=[]
for path in sorted((PROJECT_ROOT/'data/derived/sofa').glob('*/60_features/sepsis3_development_features.manifest.json')):
    try:
        manifest=ArtifactStore(path.parent).validate('sepsis3_development_features',expected_config=artifact_config); valid.append((manifest.created_at_utc,path.parents[1]))
    except (FileNotFoundError,ArtifactValidationError): pass
if not valid: raise RuntimeError('Ejecute primero los notebooks 00–12.')
RUN_ROOT=sorted(valid,key=lambda x:(x[0],str(x[1])))[-1][1]
landmarks=ArtifactStore(RUN_ROOT/'50_landmarks'); features=ArtifactStore(RUN_ROOT/'60_features'); cohort_store=ArtifactStore(RUN_ROOT/'00_cohort')
cohort=cohort_store.read_dataframe('cohort_stays',expected_config=artifact_config)
source_artifacts={'cohort_stays':cohort_store.validate('cohort_stays',expected_config=artifact_config).sha256}
for partition in ('development','validation'):
    source_artifacts[f'{partition}_landmarks']=landmarks.validate(f'sepsis3_{partition}_landmarks',expected_config=artifact_config).sha256
    source_artifacts[f'{partition}_features']=features.validate(f'sepsis3_{partition}_features',expected_config=artifact_config).sha256
implementation_hash=implementation_sha256([PROJECT_ROOT/'src/mimic_sepsis'/name for name in ('artifacts.py','evaluation.py','missingness.py','modeling.py','report_artifacts.py','reporting.py','subgroups.py')])
report_config={'report_schema_version':1,'source_run':RUN_ROOT.name,'source_artifacts':source_artifacts,'modeling':model_config,'evaluation':evaluation_config,'subgroups':subgroup_config,'implementation_sha256':implementation_hash}
REPORT_ROOT=PROJECT_ROOT/'data/derived/pretest_reports'/canonical_sha256(report_config)[:16]
print(f'Ejecución validada: {RUN_ROOT.name}')

## Muestras observadas y modelos comparables

In [ ]:
def load(partition):
    return assemble_modeling_table(
        landmarks.read_dataframe(f'sepsis3_{partition}_landmarks',expected_config=artifact_config),
        features.read_dataframe(f'sepsis3_{partition}_features',expected_config=artifact_config),
        horizon_hours=model_config['primary_horizon_hours'])
development=load('development'); validation=load('validation')
try:
    report,report_manifest=read_aggregate_report(REPORT_ROOT,expected_config=report_config)
    print('Informe agregado verificado y reutilizado.')
except (FileNotFoundError,ArtifactValidationError):
    report=build_development_report(
        development,validation,feature_columns=model_config['clinical_baseline_features'],
        folds=model_config['cross_validation_folds'],logistic_c=model_config['logistic_c'],
        bootstrap_replicates=evaluation_config['bootstrap_replicates'],
        confidence_level=evaluation_config['confidence_level'],seed=model_config['seed'],
        exploratory_thresholds=evaluation_config['exploratory_threshold_probabilities'],
        cohort=cohort,subgroup_columns=subgroup_config['columns'],
        subgroup_minimum_events=subgroup_config['minimum_events'],
        subgroup_minimum_nonevents=subgroup_config['minimum_nonevents'],
        privacy_minimum_cell=subgroup_config['privacy_minimum_cell'],
        nonlinear_pipeline=nonlinear_pipeline)
    report_manifest=write_aggregate_report(report,REPORT_ROOT,data_version=artifact_config['data_release'],code_version=implementation_hash[:12],config=report_config)
    print('Informe agregado calculado y persistido.')
print(f'Report SHA-256: {report_manifest.report_sha256}')
display(report.sample_flow)
display(report.point_metrics)
display(report.metric_intervals)
display(report.paired_intervals)
display(report.threshold_metrics)
display(report.subgroup_performance)
display(report.missingness_summary)
recalibration_ready=calibration_readiness(validation,minimum_event_patients=calibration_config['minimum_event_patients'],minimum_nonevent_patients=calibration_config['minimum_nonevent_patients'],privacy_minimum_cell=subgroup_config['privacy_minimum_cell'])
display(pd.DataFrame([{**recalibration_ready.to_dict(),'fit_partition':calibration_config['fit_partition'],'method':calibration_config['method']}]))

## Comparación pareada con ggplot2

In [ ]:
rscript=shutil.which('Rscript')
if not rscript: raise RuntimeError('Rscript no está disponible en este kernel.')
figure_dir=PROJECT_ROOT/'reports/figures'; figure_dir.mkdir(parents=True,exist_ok=True)
csv=PROJECT_ROOT/'data/derived/pretest_intervals.csv'; dca_csv=PROJECT_ROOT/'data/derived/pretest_decision_curve.csv'; missing_csv=PROJECT_ROOT/'data/derived/pretest_missingness.csv'; csv.parent.mkdir(parents=True,exist_ok=True)
svg=figure_dir/'pretest_model_comparison.svg'; dca_svg=figure_dir/'pretest_decision_curve.svg'; missing_svg=figure_dir/'pretest_missingness.svg'; script=PROJECT_ROOT/'data/derived/pretest_plot.R'
report.paired_intervals.to_csv(csv,index=False)
report.decision_curves.to_csv(dca_csv,index=False)
report.missingness_summary.to_csv(missing_csv,index=False)
script.write_text("""args <- commandArgs(trailingOnly=TRUE)
suppressPackageStartupMessages(library(ggplot2))
d <- read.csv(args[1]); d$metric <- factor(d$metric, levels=rev(unique(d$metric)))
p <- ggplot(d,aes(estimate,metric,colour=comparison)) + geom_vline(xintercept=0,linetype=2,colour='grey50') + geom_errorbar(aes(xmin=lower,xmax=upper),orientation='y',width=.18,position=position_dodge(width=.55)) + geom_point(position=position_dodge(width=.55),size=2) + facet_wrap(~sample) + labs(title='Comparaciones preespecificadas de modelos',subtitle='Diferencias con IC percentil; bootstrap por paciente; test cerrado',x='Diferencia: segundo modelo menos primero',y=NULL,colour='Comparación') + theme_minimal(base_size=11)
ggsave(args[2],p,width=10,height=5.5,device=grDevices::svg)
dca <- read.csv(args[3]); dca$strategy <- factor(dca$strategy,levels=c('gradient_boosting','logistic','reference','treat_all','treat_none'))
p2 <- ggplot(dca,aes(threshold,net_benefit,colour=strategy,linetype=strategy)) + geom_hline(yintercept=0,colour='grey70') + geom_line(linewidth=.8) + facet_wrap(~sample,scales='free_y') + labs(title='Decision-curve exploratoria',subtitle='Sin acción clínica ni umbral operativo aprobados; test cerrado',x='Umbral de probabilidad',y='Beneficio neto por landmark',colour='Estrategia',linetype='Estrategia') + theme_minimal(base_size=11)
ggsave(args[4],p2,width=10,height=5.5,device=grDevices::svg)
m <- read.csv(args[5]); m <- m[tolower(as.character(m$privacy_suppressed)) != 'true',]; m$feature <- factor(m$feature,levels=rev(unique(m$feature)))
p3 <- ggplot(m,aes(missing_fraction,feature,colour=sample)) + geom_point(position=position_dodge(width=.45),size=2.4) + scale_x_continuous(labels=scales::percent_format(accuracy=1)) + labs(title='Disponibilidad de predictores del baseline',subtitle='Fracción ausente por landmark; celdas pequeñas suprimidas',x='Landmarks sin valor',y=NULL,colour='Muestra') + theme_minimal(base_size=11)
ggsave(args[6],p3,width=10,height=5.5,device=grDevices::svg)
""",encoding='utf-8')
result=subprocess.run([rscript,str(script),str(csv),str(svg),str(dca_csv),str(dca_svg),str(missing_csv),str(missing_svg)],capture_output=True,text=True)
if result.returncode: raise RuntimeError(result.stderr)
display(SVG(filename=str(svg)))
display(SVG(filename=str(dca_svg)))
display(SVG(filename=str(missing_svg)))

## Criterio de cierre

El informe es reproducible si los manifiestos validan, development/validation se ensamblan sin pérdidas, las réplicas agrupadas se contabilizan y las tres figuras se generan con `ggsave()`. La decision-curve es exploratoria porque todavía no existe acción clínica ni umbral operativo aprobado. Missingness se audita como disponibilidad y estrato descriptivo, nunca como resultado. La calibración solo se ajustará en validation después de congelar el modelo y superar los mínimos de pacientes; el demo debe quedar bloqueado. No se selecciona un modelo por resultados del demo ni se abre el test. El informe final clínico requiere MIMIC-IV completo, decisiones firmadas y un único acceso registrado al test congelado.